In [2]:
import pandas as pd
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [3]:
def load_data(filepath):
    """Step 1: Load dataset"""
    df = pd.read_csv(filepath)
    print(f"Loaded dataset: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    return df

In [4]:
def explore_data(df, text_col='review_text', rating_col='review_rating'):
    """Step 2: Data Exploration"""
    print("\nSample Reviews:")
    print(df[[text_col, rating_col]].head(5).to_string())

    print(f"\nRating Distribution:")
    print(df[rating_col].value_counts().sort_index())

    print(f"\nNull values in '{text_col}': {df[text_col].isnull().sum()}")
    print(f"Duplicates: {df[text_col].duplicated().sum()}")
    print(f"Very short reviews (< 3 words): {df[text_col].str.split().str.len().lt(3).sum()}")
    print(f"Whitespace-only: {df[text_col].str.strip().eq('').sum()}")
    return df

In [5]:
def clean_data(df, text_col='review_text'):
    """Step 3: Data Cleaning — remove null/empty/duplicate rows"""
    before = len(df)
    df = df[df[text_col].notna()].copy()
    df = df[df[text_col].str.strip() != '']
    df = df.drop_duplicates(subset=text_col, keep='first')
    print(f"\nRemoved {before - len(df)} rows | Remaining: {len(df)}")
    return df

In [6]:
def preprocess_text(text):
    """Helper: Clean individual review text"""
    text = str(text).strip()
    if text.lower() in ['null', 'none', 'nan', 'n/a', '']:
        return ''
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_data(df, text_col='review_text'):
    """Step 4: Text Preprocessing"""
    df['review_text_clean'] = df[text_col].apply(preprocess_text)

    empty = df['review_text_clean'].str.strip().eq('').sum()
    if empty > 0:
        df = df[df['review_text_clean'].str.strip() != '']
        print(f"Removed {empty} string-null reviews after preprocessing")

    print(f"\nPreprocessing complete | Total reviews: {len(df)}")
    print(f"Sample: {df['review_text_clean'].iloc[0][:100]}...")
    return df

In [7]:
def vader_sentiment(df, text_col='review_text_clean'):
    """Step 5: VADER Sentiment Scoring"""
    sia = SentimentIntensityAnalyzer()

    df['vader_scores']    = df[text_col].apply(lambda x: sia.polarity_scores(x))
    df['vader_neg']       = df['vader_scores'].apply(lambda x: x['neg'])
    df['vader_neu']       = df['vader_scores'].apply(lambda x: x['neu'])
    df['vader_pos']       = df['vader_scores'].apply(lambda x: x['pos'])
    df['vader_compound']  = df['vader_scores'].apply(lambda x: x['compound'])

    df['vader_sentiment'] = df['vader_compound'].apply(
        lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
    )

    print(f"\nVADER Scoring complete!")
    print(f"\nSentiment Distribution:")
    print(df['vader_sentiment'].value_counts())
    return df

In [9]:
def run_pipeline(filepath, text_col='review_text', rating_col='review_rating'):
    """Master pipeline — runs all steps in order"""
    df = load_data(filepath)
    df = explore_data(df, text_col, rating_col)
    df = clean_data(df, text_col)
    df = preprocess_data(df, text_col)
    df = vader_sentiment(df, text_col='review_text_clean')
    return df

# RUN
df = run_pipeline('restaurants_combined_reviews_final.csv')

Loaded dataset: (1173, 12)
Columns: ['place_id', 'restaurant_name', 'address', 'latitude', 'longitude', 'restaurant_rating', 'user_rating_count', 'review_text', 'review_rating', 'review_time', 'author_name', 'area']

Sample Reviews:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [10]:
df

,place_id,restaurant_name,address,latitude,longitude,restaurant_rating,user_rating_count,review_text,review_rating,review_time,author_name,area,review_text_clean,vader_scores,vader_neg,vader_neu,vader_pos,vader_compound,vader_sentiment
0,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,"Devkota Sadak, Kathmandu 44600, Nepal",27.692262,85.336472,4.5,1180.0,The food was delicious and tasty. staff was ve...,5.0,2025-10-24T10:00:20.000207376Z,Kabir Bhattrai,Baneshwor,The food was delicious and tasty. staff was ve...,"{'neg': 0.0, 'neu': 0.672, 'pos': 0.328, 'comp...",0.000,0.672,0.328,0.8514,positive
1,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,"Devkota Sadak, Kathmandu 44600, Nepal",27.692262,85.336472,4.5,1180.0,Its honestly one of the prettiest spots I’ve b...,5.0,2025-08-09T06:42:53.288050672Z,Nirajan Neupane,Baneshwor,Its honestly one of the prettiest spots Ive be...,"{'neg': 0.039, 'neu': 0.713, 'pos': 0.248, 'co...",0.039,0.713,0.248,0.9287,positive
2,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,"Devkota Sadak, Kathmandu 44600, Nepal",27.692262,85.336472,4.5,1180.0,It was my first time visit @Drishya lounge and...,5.0,2025-05-22T10:26:00.123473Z,Kranti Dhakal,Baneshwor,It was my first time visit @Drishya lounge and...,"{'neg': 0.0, 'neu': 0.593, 'pos': 0.407, 'comp...",0.000,0.593,0.407,0.9650,positive
3,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,"Devkota Sadak, Kathmandu 44600, Nepal",27.692262,85.336472,4.5,1180.0,I had the pleasure of dining at Drishya Lounge...,5.0,2025-03-07T17:43:42.037954Z,Avishek Rouniyar,Baneshwor,I had the pleasure of dining at Drishya Lounge...,"{'neg': 0.012, 'neu': 0.733, 'pos': 0.255, 'co...",0.012,0.733,0.255,0.9959,positive
4,ChIJ5SO0EQ0Z6zkRTMnqUebrTqA,Drishya Lounge - Best Lounge in New Baneshwor,"Devkota Sadak, Kathmandu 44600, Nepal",27.692262,85.336472,4.5,1180.0,"I recently visited Drishya Lounge, and I must ...",5.0,2025-01-29T10:43:38.655406Z,Riya Subedi,Baneshwor,"I recently visited Drishya Lounge, and I must ...","{'neg': 0.0, 'neu': 0.78, 'pos': 0.22, 'compou...",0.000,0.780,0.220,0.9690,positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1168,ChIJQx0tsK8a6zkRVfsW8Wg0xBM,Sakotha Woh-Chatamari Fast Food,MCCH+PQH Chatubrahma MahaVihar (Tadhichen Baha...,27.671905,85.429407,4.6,24.0,This place serves only baara and potatoes. The...,5.0,2019-07-14T12:56:08.851456Z,dipesh raj manandhar,Bhaktapur Durbar Square,This place serves only baara and potatoes. The...,"{'neg': 0.025, 'neu': 0.846, 'pos': 0.129, 'co...",0.025,0.846,0.129,0.7686,positive
1169,ChIJQx0tsK8a6zkRVfsW8Wg0xBM,Sakotha Woh-Chatamari Fast Food,MCCH+PQH Chatubrahma MahaVihar (Tadhichen Baha...,27.671905,85.429407,4.6,24.0,Opens up around 1 or 2pm. Just bara (lentil pa...,5.0,2025-10-17T09:15:20.986774022Z,Sandeep Gyawali,Bhaktapur Durbar Square,Opens up around 1 or 2pm. Just bara (lentil pa...,"{'neg': 0.03, 'neu': 0.864, 'pos': 0.106, 'com...",0.030,0.864,0.106,0.7343,positive
1170,ChIJQx0tsK8a6zkRVfsW8Wg0xBM,Sakotha Woh-Chatamari Fast Food,MCCH+PQH Chatubrahma MahaVihar (Tadhichen Baha...,27.671905,85.429407,4.6,24.0,This place is famous for newari dish Baara and...,4.0,2022-06-19T15:43:11.661521Z,saru shrestha,Bhaktapur Durbar Square,This place is famous for newari dish Baara and...,"{'neg': 0.0, 'neu': 0.777, 'pos': 0.223, 'comp...",0.000,0.777,0.223,0.9144,positive
1171,ChIJQx0tsK8a6zkRVfsW8Wg0xBM,Sakotha Woh-Chatamari Fast Food,MCCH+PQH Chatubrahma MahaVihar (Tadhichen Baha...,27.671905,85.429407,4.6,24.0,This place was awesome. Food was freshly made ...,5.0,2025-08-31T13:03:19.777128580Z,John Dellamore,Bhaktapur Durbar Square,This place was awesome. Food was freshly made ...,"{'neg': 0.0, 'neu': 0.837, 'pos': 0.163, 'comp...",0.000,0.837,0.163,0.6249,positive
